# Compare cropping intensity and drought

See how cropping intensity and moderate-to-severe drought vary by year and across micro-watersheds without turning association into a causal claim.

Run each cell with **Shift+Enter**. The location controls default to the active KYL tehsil when this notebook is downloaded from CoRE Stack.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"cropping_intensity\",\"label\":\"Cropping Intensity\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"crop_intensity\",\"layerNameTemplate\":\"{district}_{tehsil}_intensity\",\"period\":\"2017 to 2024\",\"description\":\"Annual cropping intensity and single-, double-, and triple-cropped area.\"},{\"id\":\"drought\",\"label\":\"Drought\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"drought\",\"layerNameTemplate\":\"{district}_{tehsil}_drought\",\"period\":\"2017 to 2024\",\"description\":\"Dry spells and weekly mild, moderate, and severe drought indicators.\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

state_input = widgets.Text(value=SCOPE["state"], description="State:", layout=widgets.Layout(width="98%"))
district_input = widgets.Text(value=SCOPE["district"], description="District:", layout=widgets.Layout(width="98%"))
tehsil_input = widgets.Text(value=SCOPE["tehsil"], description="Tehsil:", layout=widgets.Layout(width="98%"))
display(widgets.VBox([
    widgets.HTML("<b>Study location</b><br><small>Change a name here, then rerun the data cells. No Python editing is needed.</small>"),
    state_input, district_input, tehsil_input,
]))

def selected_scope():
    return {
        "state": state_input.value.strip(),
        "district": geoserver_name(district_input.value),
        "tehsil": geoserver_name(tehsil_input.value),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
YEARS = list(range(2017, 2025))

def agriculture_profile(crop_frame, drought_frame):
    crop = with_uid(crop_frame).set_index("uid")
    drought = with_uid(drought_frame).set_index("uid")
    profile = pd.DataFrame(index=crop.index.intersection(drought.index))
    for year in YEARS:
        profile[f"crop_{year}"] = pd.to_numeric(crop.get(f"cropping_intensity_{year}"), errors="coerce")
        moderate = pd.to_numeric(drought.get(f"w_mod_{year}"), errors="coerce")
        severe = pd.to_numeric(drought.get(f"w_sev_{year}"), errors="coerce")
        profile[f"drought_{year}"] = moderate.add(severe, fill_value=np.nan)
    profile["Mean cropping intensity"] = profile[[f"crop_{year}" for year in YEARS]].mean(axis=1)
    profile["Mean moderate + severe drought weeks"] = profile[[f"drought_{year}" for year in YEARS]].mean(axis=1)
    profile.index.name = "MWS UID"
    return profile

def yearly_agriculture(profile):
    return pd.DataFrame({
        "Year": YEARS,
        "Mean cropping intensity": [profile[f"crop_{year}"].mean() for year in YEARS],
        "MWS with 5+ drought weeks (%)": [(profile[f"drought_{year}"] >= 5).mean() * 100 for year in YEARS],
    })

def plot_yearly_agriculture(summary):
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    axes[0].plot(summary["Year"], summary["Mean cropping intensity"], marker="o", color="#15803d")
    axes[0].set(title="Tehsil mean cropping intensity", ylabel="Cropping intensity")
    axes[1].bar(summary["Year"], summary["MWS with 5+ drought weeks (%)"], color="#dc2626")
    axes[1].set(title="MWSes meeting the published drought-year threshold", ylabel="Share of MWS (%)", xlabel="Year")
    plt.tight_layout()
    plt.show()

def plot_agriculture_association(profile):
    clean = profile[["Mean moderate + severe drought weeks", "Mean cropping intensity"]].dropna()
    correlation = clean.corr().iloc[0, 1] if len(clean) > 1 else np.nan
    plt.figure(figsize=(8, 5))
    plt.scatter(clean.iloc[:, 0], clean.iloc[:, 1], color="#7c3aed", alpha=0.75)
    plt.xlabel("Mean moderate + severe drought weeks")
    plt.ylabel("Mean cropping intensity")
    plt.title(f"Within-tehsil association · correlation {correlation:.2f}")
    plt.show()
    return clean, correlation

## 1. Join the two MWS layers

The published MWS UID keeps every comparison at the same spatial unit.

In [ ]:
crop_geojson = await load_geojson("cropping_intensity")
drought_geojson = await load_geojson("drought")
crop_frame = to_frame(crop_geojson)
drought_frame = to_frame(drought_geojson)
profile = agriculture_profile(crop_frame, drought_frame)
summary = yearly_agriculture(profile)
display(summary.round(2))

## 2. Compare years without hiding the two scales

Separate panels avoid implying that cropping intensity and drought percentage use the same units.

In [ ]:
plot_yearly_agriculture(summary)

## 3. Inspect association across MWSes

The printed correlation is descriptive for this tehsil and period. It is not an estimate of drought's causal effect.

In [ ]:
association, correlation = plot_agriculture_association(profile)
print(f"Compared {len(association)} MWSes; descriptive correlation = {correlation:.2f}.")

## 4. Map a transparent shortlist

This shortlist means above-upper-quartile drought weeks and below-median cropping intensity; the thresholds are printed.

In [ ]:
drought_cut = profile["Mean moderate + severe drought weeks"].quantile(0.75)
crop_cut = profile["Mean cropping intensity"].median()
shortlist = profile[(profile["Mean moderate + severe drought weeks"] >= drought_cut) &
                    (profile["Mean cropping intensity"] <= crop_cut)]
print(f"Thresholds: drought ≥ {drought_cut:.2f} weeks; cropping intensity ≤ {crop_cut:.2f}.")
display(shortlist.iloc[:, -2:].round(2))
show_on_map("agriculture-shortlist", features_for_uids(crop_geojson, shortlist.index),
            "Notebook · drought/cropping shortlist", fillColor="#f97316", strokeColor="#7c2d12", fillOpacity=0.6)

## Interpretation

This notebook can reveal years or MWSes worth investigating. Rainfall, irrigation, soils, crop choice, infrastructure, markets, and data quality can all affect the observed relationship.

## Optional: inspect annual values for one MWS

In [ ]:
mws_picker = widgets.Dropdown(options=sorted(profile.index), description="MWS:")
display(mws_picker)
display(profile.loc[[mws_picker.value]].round(2))